# 03: Train a model

One training run: config → data → train → run folder. Defaults reproduce the dissertation's final
model (P0: DGCNN + FoldingNet, completing plane-cut teeth).

**Run it in Colab on a GPU runtime.** One run takes about 20–70 minutes on an A100; on a CPU it
would take days, so section 3 refuses to train on a CPU unless `TRAIN_ON_CPU = True`.

**Data.** Upload the processed datasets (`complete_rebuilt/` and `partial_rebuilt/` from your local
`data/processed/`) to `MyDrive/dental-3d-vae/data/processed/`. Teeth3DS+ is licensed: keep it in your
own Drive and **clear this notebook's outputs before committing**.

**Outputs** go to a new run folder (`MyDrive/dental-3d-vae/runs/` in Colab): `config.json`,
`environment.json`, `metrics.jsonl` (one line per epoch), `model.pt` (the best model, saved as soon as
it is found) and `summary.json`.

In [ ]:
# Colab setup: fetch this repository's latest code from GitHub. Does nothing locally.
# Same cell in every notebook. After pushing new code: Runtime > Restart session, then run all cells.
import sys

REPO = "dgelbier23/dental-3d-vae"

if "google.colab" in sys.modules:
    import subprocess
    from pathlib import Path
    from google.colab import userdata

    try:  # A token is only needed while the repository is private.
        token = userdata.get("GITHUB_TOKEN")
    except Exception:  # No such secret, or this notebook may not read it.
        token = ""
    login = f"x-access-token:{token}@" if token else ""
    repo_dir = Path("/content") / REPO.split("/")[1]
    subprocess.run(["rm", "-rf", str(repo_dir)], check=True)
    result = subprocess.run(["git", "clone", "--depth", "1", f"https://{login}github.com/{REPO}.git", str(repo_dir)],
                            capture_output=True, text=True)
    if result.returncode:
        error = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(f"{error} If the repository is private, add a GITHUB_TOKEN Colab secret (see the README).")
    if token:  # git clone saves the URL it was given, so remove the token from the clone's settings.
        subprocess.run(["git", "-C", str(repo_dir), "remote", "set-url", "origin",
                        f"https://github.com/{REPO}.git"], check=True)
    sys.path.insert(0, str(repo_dir / "src"))

## 1. Config

Every setting of the run. To try a variant, change a field and give the run a new `run_name`, e.g.
`ExperimentConfig(run_name="B4_beta_0.01", reconstruction_mode="complete", kl_weight=0.01)`.

In [ ]:
from dataclasses import asdict

from dental_3d_vae.experiments.config import ExperimentConfig
from dental_3d_vae.runtime.env import get_data_dir
from dental_3d_vae.viz.plotting import use_theme

use_theme()
config = ExperimentConfig()  # P0
COMPLETE_DIR = get_data_dir() / "processed" / "complete_rebuilt"
PARTIAL_DIR = get_data_dir() / "processed" / "partial_rebuilt"
TRAIN_ON_CPU = False  # Training on a CPU takes days; leave False unless testing a tiny config.
asdict(config)

## 2. Setup

In Colab the datasets are first copied from Drive to the runtime's own disk (about 110 MB, seconds):
the loaders read one tooth at a time, which is slow from Drive. The model is built right after
seeding, so its starting weights depend only on `config.seed`.

In [ ]:
from dental_3d_vae.experiments.runs import create_run_dir
from dental_3d_vae.ml.data import make_loaders
from dental_3d_vae.ml.models.vae import build_vae
from dental_3d_vae.runtime.device import get_device, set_seed
from dental_3d_vae.runtime.env import copy_to_runtime, get_runs_dir

COMPLETE_DIR, PARTIAL_DIR = copy_to_runtime(COMPLETE_DIR), copy_to_runtime(PARTIAL_DIR)

set_seed(config.seed, config.deterministic)  # Before any CUDA work.
device = get_device()
loaders = make_loaders(config, COMPLETE_DIR, PARTIAL_DIR)
model = build_vae(config.encoder, config.decoder, config.latent_dim, config.num_points)
print(f"Device: {device}")
print(f"Model: {config.encoder} + {config.decoder}, {sum(p.numel() for p in model.parameters()):,} parameters")
print("Teeth:", {split: len(loader.dataset) for split, loader in loaders.items()})

## 3. Train

Each epoch trains on sampled latents (loss: Chamfer distance + β·KL, plus the T-Net regulariser for
`tnet_pointnet`), then scores the validation set with z = μ. Training stops after `patience` epochs
without a better validation Chamfer distance. Every epoch is logged and every new best model saved
straight away, so a Colab disconnect loses at most the epoch in progress.

In [ ]:
from dental_3d_vae.ml.train import fit

if device.type == "cpu" and not TRAIN_ON_CPU:
    raise RuntimeError("No GPU: use a Colab GPU runtime, or set TRAIN_ON_CPU = True for a tiny test config.")
run_dir = create_run_dir(get_runs_dir(), config)
print(f"Run folder: {run_dir}")
summary = fit(model, loaders, config, device, run_dir)
summary

## 4. Training curves

Training Chamfer distance comes from sampled latents and validation from z = μ, so validation can sit
below training. The dotted line marks the best epoch: the saved model. Test-set metrics (CD, MRRE,
ORPE, SCE) are computed in the evaluation notebook.

In [ ]:
from dental_3d_vae.experiments.runs import load_metrics
from dental_3d_vae.viz.plotting import plot_training_history

plot_training_history(load_metrics(run_dir)).show()